In [5]:
import boto3, os
from io import StringIO
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

s3 = boto3.client("s3",
    region_name="eu-west-3",
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
)

# --- CSV: Read products.csv ---
response = s3.get_object(Bucket="kickz-empire-data", Key="raw/orders/orders.csv")
df = pd.read_csv(StringIO(response["Body"].read().decode("utf-8")))

print(df.shape)        # rows × columns
print(df.dtypes)       # column types
print(df.head())       # first rows
print(df.describe())   # statistics

(17072, 31)
order_id                      str
order_uuid                    str
user_id                       str
email                         str
order_date                    str
order_date_epoch            int64
status                        str
num_items                   int64
subtotal_usd              float64
shipping_method               str
shipping_cost_usd         float64
coupon_code                   str
discount_amount_usd       float64
tax_usd                   float64
total_usd                 float64
currency                      str
payment_method                str
billing_country               str
billing_city                  str
billing_postal_code         int64
shipping_country              str
shipping_city                 str
shipping_postal_code        int64
ip_address                    str
user_agent                    str
_stripe_charge_id             str
_stripe_payment_intent        str
_paypal_txn_id                str
_internal_order_flags       int64
_f

In [2]:
from io import StringIO

# --- JSONL: Read reviews.jsonl ---
response = s3.get_object(Bucket="kickz-empire-data", Key="raw/reviews/reviews.jsonl")
jsonl_content = response["Body"].read().decode("utf-8")

# pd.read_json() with lines=True reads one JSON object per line
df_reviews = pd.read_json(StringIO(jsonl_content), lines=True)

print(df_reviews.shape)
print(df_reviews.dtypes)
print(df_reviews.head())

(2930, 20)
review_id                             str
product_id                            str
product_name                          str
user_id                               str
user_name                             str
rating                              int64
title                                 str
body                                  str
verified_purchase                    bool
submitted_at          datetime64[us, UTC]
moderation_status                     str
moderated_at          datetime64[us, UTC]
helpful_votes                       int64
reported                             bool
photos_count                        int64
_moderation_score                 float64
_sentiment_raw                    float64
_toxicity_score                   float64
_language_detected                    str
_review_source                        str
dtype: object
                              review_id product_id  \
0  16336ca4-ccfd-4089-999d-19cb0960f2a7   KE-10062   
1  46cfa7c2-7a8e-491f-b15f-

In [3]:
from io import BytesIO
import pyarrow.parquet as pq

# --- Parquet: Read a single partition file ---
response = s3.get_object(
    Bucket="kickz-empire-data",
    Key="raw/clickstream/dt=2026-02-05/part-00001.snappy.parquet"
)
table = pq.read_table(BytesIO(response["Body"].read()))
df_click = table.to_pandas()

print(df_click.shape)
print(df_click.dtypes)
print(df_click.head())

(5000, 27)
event_id                   str
event_type                 str
timestamp                  str
timestamp_epoch_ms       int64
user_id                    str
session_id                 str
page_url                   str
page_path                  str
page_type                  str
referrer_url               str
referrer_source            str
user_agent_raw             str
ip_address                 str
viewport_width           int64
viewport_height          int64
screen_color_depth       int64
device_pixel_ratio     float64
accept_language            str
is_bot                    bool
_ga_client_id              str
_gtm_container_id          str
_dom_interactive_ms      int64
_dom_complete_ms         int64
_ttfb_ms                 int64
_connection_type           str
_js_heap_size_mb       float64
_consent_string            str
dtype: object
                               event_id event_type  \
0  ac0e758c-cc0b-4f45-b1c7-2f63dc1a0af1   pageview   
1  7d534b62-0fae-493c-897b-3dd